# ResNet50 from Scratch for Multi-Class Medical Image Classification

## ISIC 2019 Skin Lesion Dataset

This notebook trains a **ResNet50 model from scratch** for classification of 8 skin lesion categories from the ISIC 2019 dataset.

### Experiment Configuration

- **Architecture:** ResNet50
- **Dataset:** ISIC 2019
- **Number of classes:** 8
- **Training strategy:** From scratch
- **Pretrained weights:** None
- **Transfer learning:** No
- **Data augmentation:** No
- **Input size:** 224 × 224
- **Framework:** PyTorch
- **Hardware acceleration:** Apple MPS when available
- **Loss:** Class-weighted Cross Entropy
- **Optimizer:** AdamW
- **Evaluation metrics:** Accuracy, Precision, Recall, F1-score, Macro F1 and Confusion Matrix

The permanent lesion-aware train, validation and test splits created in `00_Dataset_Preparation.ipynb` are reused to ensure a fair comparison with all later experiments.

In [ ]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path
import random
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import resnet50

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("PyTorch version:", torch.__version__)
print("Libraries imported successfully.")

In [ ]:
# ============================================================
# Project Configuration
# ============================================================

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "dataset"
SPLIT_DIR = DATA_DIR / "splits"

TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "val.csv"
TEST_CSV = SPLIT_DIR / "test.csv"

MODEL_DIR = PROJECT_DIR / "saved_models"
RESULTS_DIR = PROJECT_DIR / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 8
IMAGE_SIZE = 224
RANDOM_SEED = 42

print("Project directory :", PROJECT_DIR)

print("\nDataset split verification:")
print("Train split exists :", TRAIN_CSV.exists())
print("Val split exists   :", VAL_CSV.exists())
print("Test split exists  :", TEST_CSV.exists())

print("\nExperiment configuration:")
print("Number of classes :", NUM_CLASSES)
print("Input image size   :", IMAGE_SIZE)
print("Random seed        :", RANDOM_SEED)

In [ ]:
# ============================================================
# Reproducibility
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Random seed set to:", RANDOM_SEED)

In [ ]:
# ============================================================
# Device Configuration
# ============================================================

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using Apple Silicon GPU via MPS.")

elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("Using CUDA GPU.")

else:
    DEVICE = torch.device("cpu")
    print("Using CPU.")

print("Selected device:", DEVICE)

In [ ]:
# ============================================================
# Load Train / Validation / Test Splits
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("=" * 60)
print("DATASET SPLITS")
print("=" * 60)

print(f"Training images   : {len(train_df):,}")
print(f"Validation images : {len(val_df):,}")
print(f"Test images       : {len(test_df):,}")

total_images = (
    len(train_df)
    + len(val_df)
    + len(test_df)
)

print(f"Total images      : {total_images:,}")

In [ ]:
# ============================================================
# Class Information
# ============================================================

class_info = (
    train_df[
        ["class_index", "class_code", "class_name"]
    ]
    .drop_duplicates()
    .sort_values("class_index")
    .reset_index(drop=True)
)

print("Classes used by the model:\n")

for _, row in class_info.iterrows():
    print(
        f"{int(row['class_index'])} -> "
        f"{row['class_code']:5s} -> "
        f"{row['class_name']}"
    )

In [ ]:
# ============================================================
# Resized Image Cache Configuration
# ============================================================

CACHE_SIZE = 256

CACHE_DIR = DATA_DIR / f"ISIC_2019_{CACHE_SIZE}"

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

existing_cached_images = list(
    CACHE_DIR.glob("*.jpg")
)

print("Cache directory:")
print(CACHE_DIR)

print("\nCache size       :", CACHE_SIZE)
print("Cached images   :", len(existing_cached_images))
print("Expected images :", len(train_df) + len(val_df) + len(test_df))

In [ ]:
# ============================================================
# Build / Resume 256x256 Image Cache
# ============================================================

all_data_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

failed_resizes = []

cache_start_time = time.time()

for index, row in all_data_df.iterrows():

    source_path = Path(row["image_path"])
    destination_path = CACHE_DIR / source_path.name

    # Skip images already processed
    if destination_path.exists():
        continue

    try:
        with Image.open(source_path) as image:

            image = image.convert("RGB")

            image = image.resize(
                (CACHE_SIZE, CACHE_SIZE),
                Image.Resampling.LANCZOS
            )

            image.save(
                destination_path,
                format="JPEG",
                quality=95
            )

    except Exception as error:
        failed_resizes.append(
            (source_path.name, str(error))
        )

    if (index + 1) % 2500 == 0:

        elapsed = time.time() - cache_start_time

        print(
            f"Processed {index + 1:,} / "
            f"{len(all_data_df):,} images "
            f"| {elapsed / 60:.2f} min"
        )


cache_time = time.time() - cache_start_time

print("\n" + "=" * 60)
print("IMAGE CACHE PROCESSING COMPLETE")
print("=" * 60)

print("Expected images :", len(all_data_df))
print("Failed images   :", len(failed_resizes))
print(f"Time taken      : {cache_time / 60:.2f} min")

In [ ]:
# ============================================================
# Build / Resume 256x256 Image Cache
# ============================================================

all_data_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

failed_resizes = []

cache_start_time = time.time()

for index, row in all_data_df.iterrows():

    source_path = Path(row["image_path"])
    destination_path = CACHE_DIR / source_path.name

    # Skip images already processed
    if destination_path.exists():
        continue

    try:
        with Image.open(source_path) as image:

            image = image.convert("RGB")

            image = image.resize(
                (CACHE_SIZE, CACHE_SIZE),
                Image.Resampling.LANCZOS
            )

            image.save(
                destination_path,
                format="JPEG",
                quality=95
            )

    except Exception as error:
        failed_resizes.append(
            (source_path.name, str(error))
        )

    if (index + 1) % 2500 == 0:

        elapsed = time.time() - cache_start_time

        print(
            f"Processed {index + 1:,} / "
            f"{len(all_data_df):,} images "
            f"| {elapsed / 60:.2f} min"
        )


cache_time = time.time() - cache_start_time

print("\n" + "=" * 60)
print("IMAGE CACHE PROCESSING COMPLETE")
print("=" * 60)

print("Expected images :", len(all_data_df))
print("Failed images   :", len(failed_resizes))
print(f"Time taken      : {cache_time / 60:.2f} min")

In [ ]:
# ============================================================
# Verify 256x256 Image Cache
# ============================================================

cached_images = list(
    CACHE_DIR.glob("*.jpg")
)

expected_images = (
    len(train_df)
    + len(val_df)
    + len(test_df)
)

print("=" * 60)
print("IMAGE CACHE VERIFICATION")
print("=" * 60)

print("Cached images :", len(cached_images))
print("Expected      :", expected_images)

if len(cached_images) == expected_images:
    print("\n✓ Cache verification successful.")
else:
    print("\n⚠ Cache count does not match expected image count.")

In [ ]:
# ============================================================
# Add Cached Image Paths
# ============================================================

def add_cached_path(dataframe):
    dataframe = dataframe.copy()

    dataframe["cached_image_path"] = dataframe[
        "image_path"
    ].apply(
        lambda path: str(
            CACHE_DIR / Path(path).name
        )
    )

    return dataframe


train_df = add_cached_path(train_df)
val_df = add_cached_path(val_df)
test_df = add_cached_path(test_df)

print("Example cached path:")
print(train_df.iloc[0]["cached_image_path"])

print(
    "\nFile exists:",
    Path(
        train_df.iloc[0]["cached_image_path"]
    ).exists()
)

In [ ]:
# ============================================================
# Image Preprocessing — No Data Augmentation
# ============================================================

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]

base_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGE_MEAN,
        std=IMAGE_STD
    )
])

print("Preprocessing pipeline created successfully.\n")
print(base_transform)

In [ ]:
# ============================================================
# Custom ISIC Dataset
# ============================================================

class ISICDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
        )

        self.transform = transform


    def __len__(self):

        return len(self.dataframe)


    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["cached_image_path"]
        label = int(row["class_index"])

        with Image.open(image_path) as image:

            image = image.convert("RGB")

            if self.transform is not None:
                image = self.transform(image)

        return image, label


print("ISICDataset class created successfully.")

In [ ]:
# ============================================================
# Create PyTorch Dataset Objects
# ============================================================

train_dataset = ISICDataset(
    dataframe=train_df,
    transform=base_transform
)

val_dataset = ISICDataset(
    dataframe=val_df,
    transform=base_transform
)

test_dataset = ISICDataset(
    dataframe=test_df,
    transform=base_transform
)

print("=" * 60)
print("PYTORCH DATASETS")
print("=" * 60)

print(f"Training samples   : {len(train_dataset):,}")
print(f"Validation samples : {len(val_dataset):,}")
print(f"Test samples       : {len(test_dataset):,}")

In [ ]:
# ============================================================
# Verify One Processed Sample
# ============================================================

sample_image, sample_label = train_dataset[0]

print("Image tensor shape :", sample_image.shape)
print("Image tensor dtype :", sample_image.dtype)

print("Class index        :", sample_label)
print(
    "Class code         :",
    class_info.iloc[sample_label]["class_code"]
)

print(
    "Class name         :",
    class_info.iloc[sample_label]["class_name"]
)

In [ ]:
# ============================================================
# DataLoader Configuration
# ============================================================

BATCH_SIZE = 16
NUM_WORKERS = 0

print("DataLoader configuration:")
print("Batch size  :", BATCH_SIZE)
print("Num workers :", NUM_WORKERS)

In [ ]:
# ============================================================
# Create DataLoaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("DataLoaders created successfully.\n")

print("Training batches   :", len(train_loader))
print("Validation batches :", len(val_loader))
print("Test batches       :", len(test_loader))

In [ ]:
# ============================================================
# Verify One Training Batch
# ============================================================

images, labels = next(iter(train_loader))

print("Batch image shape :", images.shape)
print("Batch label shape :", labels.shape)

print("Image dtype       :", images.dtype)
print("Label dtype       :", labels.dtype)

print("\nFirst 10 labels:")
print(labels[:10])

In [ ]:
# ============================================================
# DataLoader Speed Benchmark
# ============================================================

NUM_TEST_BATCHES = 100

benchmark_start = time.time()

for batch_index, (benchmark_images, benchmark_labels) in enumerate(train_loader):

    if batch_index + 1 >= NUM_TEST_BATCHES:
        break

benchmark_time = time.time() - benchmark_start

print("=" * 60)
print("DATALOADER SPEED BENCHMARK")
print("=" * 60)

print(f"Batches loaded   : {NUM_TEST_BATCHES}")
print(f"Images loaded    : {NUM_TEST_BATCHES * BATCH_SIZE}")
print(f"Total time       : {benchmark_time:.2f} seconds")
print(f"Time per batch   : {benchmark_time / NUM_TEST_BATCHES:.4f} seconds")

images_per_second = (
    NUM_TEST_BATCHES * BATCH_SIZE
) / benchmark_time

print(f"Images per second: {images_per_second:.2f}")

In [ ]:
# ============================================================
# Create ResNet50 From Scratch
# ============================================================

model = resnet50(
    weights=None
)

print("ResNet50 created successfully.")
print("Pretrained weights : None")
print("Transfer learning  : No")
print("Initialization     : Random")

In [ ]:
# ============================================================
# Replace Final Classification Layer
# ============================================================

in_features = model.fc.in_features

model.fc = nn.Linear(
    in_features=in_features,
    out_features=NUM_CLASSES
)

print("Original feature dimension :", in_features)
print("New output classes         :", NUM_CLASSES)

print("\nFinal classification layer:")
print(model.fc)

In [ ]:
# ============================================================
# Move Model to Selected Device
# ============================================================

model = model.to(DEVICE)

print("Model moved successfully.")
print("Selected device:", DEVICE)

In [ ]:
# ============================================================
# Model Parameter Count
# ============================================================

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

frozen_params = (
    total_params - trainable_params
)

print("=" * 60)
print("RESNET50 PARAMETER SUMMARY")
print("=" * 60)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Frozen parameters    : {frozen_params:,}")

In [ ]:
# ============================================================
# Display ResNet50 Architecture
# ============================================================

print(model)

In [ ]:
# ============================================================
# Test Forward Pass
# ============================================================

model.eval()

with torch.no_grad():

    sample_batch = images[:2].to(DEVICE)

    sample_outputs = model(sample_batch)

print("Input batch shape :", sample_batch.shape)
print("Output shape      :", sample_outputs.shape)

In [ ]:
# ============================================================
# Inspect Raw Model Outputs
# ============================================================

print("Raw logits for first image:\n")
print(sample_outputs[0])

predicted_index = sample_outputs[0].argmax().item()

print("\nPredicted class before training:")
print("Class index :", predicted_index)
print(
    "Class name  :",
    class_info.iloc[predicted_index]["class_name"]
)

In [ ]:
# ============================================================
# Calculate Class Weights
# ============================================================

class_counts = (
    train_df["class_index"]
    .value_counts()
    .sort_index()
)

total_training_samples = len(train_df)

class_weights = (
    total_training_samples
    / (NUM_CLASSES * class_counts.values)
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

print("=" * 60)
print("TRAINING CLASS DISTRIBUTION AND WEIGHTS")
print("=" * 60)

for class_index, count in class_counts.items():

    class_name = class_info.iloc[class_index]["class_name"]

    print(
        f"{class_index} -> "
        f"{class_name:28s} | "
        f"Images: {count:5,d} | "
        f"Weight: {class_weights[class_index]:.4f}"
    )

In [ ]:
# ============================================================
# Loss Function
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

print("Loss function:")
print(criterion)

print("\nUsing class-weighted Cross Entropy Loss.")

In [ ]:
# ============================================================
# Optimizer Configuration
# ============================================================

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Optimizer     : AdamW")
print("Learning rate :", LEARNING_RATE)
print("Weight decay  :", WEIGHT_DECAY)

In [ ]:
# ============================================================
# Learning Rate Scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("Scheduler        : ReduceLROnPlateau")
print("Monitors         : Validation Loss")
print("Reduction factor : 0.5")
print("Patience         : 2 epochs")

In [ ]:
# ============================================================
# Training Configuration
# ============================================================

NUM_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 5

BEST_MODEL_PATH = (
    MODEL_DIR / "resnet50_scratch_best.pth"
)

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

print("Model                  : ResNet50")
print("Training strategy      : From Scratch")
print("Pretrained weights     : None")
print("Transfer learning      : No")
print("Data augmentation      : No")

print("\nMaximum epochs         :", NUM_EPOCHS)
print("Batch size             :", BATCH_SIZE)
print("Learning rate          :", LEARNING_RATE)
print("Weight decay           :", WEIGHT_DECAY)

print("\nLoss function          : Weighted CrossEntropyLoss")
print("Optimizer              : AdamW")
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Device                 :", DEVICE)

print("\nBest model path:")
print(BEST_MODEL_PATH)

In [ ]:
# ============================================================
# Training Function — One Epoch
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    device
):
    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in dataloader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients from previous iteration
        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # -------------------------
        # Statistics
        # -------------------------

        batch_size = images.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        predictions = outputs.argmax(dim=1)

        correct_predictions += (
            predictions == labels
        ).sum().item()

        total_samples += batch_size

    epoch_loss = (
        running_loss / total_samples
    )

    epoch_accuracy = (
        correct_predictions / total_samples
    )

    return epoch_loss, epoch_accuracy


print("Training function created successfully.")

In [ ]:
# ============================================================
# Validation Function — One Epoch
# ============================================================

def validate_one_epoch(
    model,
    dataloader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Validation loss
            loss = criterion(outputs, labels)

            batch_size = images.size(0)

            running_loss += (
                loss.item() * batch_size
            )

            predictions = outputs.argmax(dim=1)

            correct_predictions += (
                predictions == labels
            ).sum().item()

            total_samples += batch_size

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    epoch_loss = (
        running_loss / total_samples
    )

    epoch_accuracy = (
        correct_predictions / total_samples
    )

    epoch_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return (
        epoch_loss,
        epoch_accuracy,
        epoch_macro_f1
    )


print("Validation function created successfully.")

In [ ]:
# ============================================================
# Initialize Training History
# ============================================================

history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": [],
    "learning_rate": [],
    "epoch_time_seconds": []
}

best_val_f1 = -1.0
epochs_without_improvement = 0

print("Training history initialized successfully.")
print("Best validation Macro F1 :", best_val_f1)
print("Early stopping counter   :", epochs_without_improvement)

In [ ]:
# ============================================================
# Single-Batch Training Sanity Check
# ============================================================

# Use evaluation mode so BatchNorm running statistics
# are not modified during this temporary test
model.eval()

sanity_images, sanity_labels = next(
    iter(train_loader)
)

sanity_images = sanity_images.to(DEVICE)
sanity_labels = sanity_labels.to(DEVICE)

optimizer.zero_grad(set_to_none=True)

# Forward pass
sanity_outputs = model(sanity_images)

# Loss
sanity_loss = criterion(
    sanity_outputs,
    sanity_labels
)

# Backward pass
sanity_loss.backward()

# Remove temporary gradients
optimizer.zero_grad(set_to_none=True)

print("=" * 60)
print("SANITY CHECK")
print("=" * 60)

print("Input shape  :", sanity_images.shape)
print("Output shape :", sanity_outputs.shape)
print("Loss         :", f"{sanity_loss.item():.4f}")

print("\n✓ Forward and backward passes completed successfully.")

In [ ]:
# ============================================================
# Batch Size 32 Compatibility Test
# ============================================================

TEST_BATCH_SIZE = 32

test_batch_loader = DataLoader(
    train_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

model.eval()

try:
    test_images, test_labels = next(
        iter(test_batch_loader)
    )

    test_images = test_images.to(DEVICE)
    test_labels = test_labels.to(DEVICE)

    optimizer.zero_grad(set_to_none=True)

    start_time = time.time()

    test_outputs = model(test_images)

    test_loss = criterion(
        test_outputs,
        test_labels
    )

    test_loss.backward()

    elapsed_time = time.time() - start_time

    optimizer.zero_grad(set_to_none=True)

    print("=" * 60)
    print("BATCH SIZE 32 TEST")
    print("=" * 60)

    print("Input shape :", test_images.shape)
    print("Output shape:", test_outputs.shape)
    print("Loss        :", f"{test_loss.item():.4f}")
    print("Step time   :", f"{elapsed_time:.2f} seconds")

    print("\n✓ Batch size 32 works successfully.")

except RuntimeError as error:

    print("⚠ Batch size 32 failed.")
    print("\nError:")
    print(error)

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

In [ ]:
# ============================================================
# Final DataLoader Configuration
# ============================================================

BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("=" * 60)
print("FINAL DATALOADER CONFIGURATION")
print("=" * 60)

print("Batch size        :", BATCH_SIZE)
print("Workers           :", NUM_WORKERS)

print("\nTraining batches   :", len(train_loader))
print("Validation batches :", len(val_loader))
print("Test batches       :", len(test_loader))

In [ ]:
# ============================================================
# Save Final Experiment Configuration
# ============================================================

experiment_config = {
    "experiment": "ResNet50_Scratch",
    "architecture": "ResNet50",
    "dataset": "ISIC_2019",
    "num_classes": NUM_CLASSES,
    "image_size": IMAGE_SIZE,
    "cache_size": CACHE_SIZE,

    "pretrained": False,
    "transfer_learning": False,
    "data_augmentation": False,

    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,

    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,

    "loss_function": "Weighted CrossEntropyLoss",

    "max_epochs": NUM_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,

    "random_seed": RANDOM_SEED,
    "device": str(DEVICE)
}

CONFIG_PATH = (
    RESULTS_DIR /
    "resnet50_scratch_config.json"
)

with open(CONFIG_PATH, "w") as file:
    json.dump(
        experiment_config,
        file,
        indent=4
    )

print("Final experiment configuration saved successfully.")
print("\nLocation:")
print(CONFIG_PATH)

print("\nBatch size recorded:", experiment_config["batch_size"])

In [ ]:
# ============================================================
# Full ResNet50 Training Loop
# ============================================================

HISTORY_PATH = (
    RESULTS_DIR /
    "resnet50_scratch_history.csv"
)

print("=" * 75)
print("STARTING RESNET50 TRAINING FROM SCRATCH")
print("=" * 75)

training_start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):

    epoch_start_time = time.time()

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    train_loss, train_accuracy = train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    (
        val_loss,
        val_accuracy,
        val_macro_f1
    ) = validate_one_epoch(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=DEVICE
    )

    # --------------------------------------------------------
    # Learning Rate Scheduler
    # --------------------------------------------------------

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # --------------------------------------------------------
    # Epoch Time
    # --------------------------------------------------------

    epoch_time = time.time() - epoch_start_time

    # --------------------------------------------------------
    # Save History
    # --------------------------------------------------------

    history["epoch"].append(epoch)

    history["train_loss"].append(
        train_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_accuracy"].append(
        val_accuracy
    )

    history["val_macro_f1"].append(
        val_macro_f1
    )

    history["learning_rate"].append(
        current_lr
    )

    history["epoch_time_seconds"].append(
        epoch_time
    )

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        HISTORY_PATH,
        index=False
    )

    # --------------------------------------------------------
    # Display Epoch Results
    # --------------------------------------------------------

    print(
        f"\nEpoch [{epoch:02d}/{NUM_EPOCHS}]"
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy * 100:.2f}%"
    )

    print(
        f"Val Loss  : {val_loss:.4f} | "
        f"Val Acc: {val_accuracy * 100:.2f}% | "
        f"Macro F1: {val_macro_f1:.4f}"
    )

    print(
        f"LR: {current_lr:.6f} | "
        f"Time: {epoch_time / 60:.2f} min"
    )

    # --------------------------------------------------------
    # Save Best Model
    # --------------------------------------------------------

    if val_macro_f1 > best_val_f1:

        best_val_f1 = val_macro_f1

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print(
            f"✓ Best model saved "
            f"(Macro F1 = {best_val_f1:.4f})"
        )

    else:

        epochs_without_improvement += 1

        print(
            "No Macro-F1 improvement for "
            f"{epochs_without_improvement} epoch(s)."
        )

    # --------------------------------------------------------
    # Early Stopping
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print("\nEarly stopping triggered.")
        break


# ============================================================
# Training Finished
# ============================================================

total_training_time = (
    time.time() -
    training_start_time
)

print("\n" + "=" * 75)
print("TRAINING COMPLETED")
print("=" * 75)

print(
    f"Epochs completed          : "
    f"{len(history['epoch'])}"
)

print(
    f"Best Validation Macro F1 : "
    f"{best_val_f1:.4f}"
)

print(
    f"Total Training Time      : "
    f"{total_training_time / 60:.2f} minutes"
)

print("\nBest model saved at:")
print(BEST_MODEL_PATH)

print("\nTraining history saved at:")
print(HISTORY_PATH)

In [ ]:
# ============================================================
# Inspect Best Training Epoch
# ============================================================

history_df = pd.read_csv(
    RESULTS_DIR / "resnet50_scratch_history.csv"
)

best_epoch_row = history_df.loc[
    history_df["val_macro_f1"].idxmax()
]

print("=" * 65)
print("BEST VALIDATION EPOCH")
print("=" * 65)

print(
    f"Epoch              : "
    f"{int(best_epoch_row['epoch'])}"
)

print(
    f"Train Loss         : "
    f"{best_epoch_row['train_loss']:.4f}"
)

print(
    f"Train Accuracy     : "
    f"{best_epoch_row['train_accuracy'] * 100:.2f}%"
)

print(
    f"Validation Loss    : "
    f"{best_epoch_row['val_loss']:.4f}"
)

print(
    f"Validation Accuracy: "
    f"{best_epoch_row['val_accuracy'] * 100:.2f}%"
)

print(
    f"Validation Macro F1: "
    f"{best_epoch_row['val_macro_f1']:.4f}"
)

print(
    f"Learning Rate      : "
    f"{best_epoch_row['learning_rate']:.8f}"
)

In [ ]:
# ============================================================
# Plot Training and Validation Loss
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Training Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet50 From Scratch - Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Plot Training and Validation Accuracy
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history_df["epoch"],
    history_df["train_accuracy"] * 100,
    label="Training Accuracy"
)

plt.plot(
    history_df["epoch"],
    history_df["val_accuracy"] * 100,
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("ResNet50 From Scratch - Training vs Validation Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Plot Validation Macro F1
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    history_df["epoch"],
    history_df["val_macro_f1"],
    label="Validation Macro F1"
)

best_epoch = int(
    history_df.loc[
        history_df["val_macro_f1"].idxmax(),
        "epoch"
    ]
)

best_f1 = history_df["val_macro_f1"].max()

plt.axvline(
    x=best_epoch,
    linestyle="--",
    label=f"Best Epoch = {best_epoch}"
)

plt.scatter(
    best_epoch,
    best_f1,
    s=80
)

plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("ResNet50 From Scratch - Validation Macro F1")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Best epoch    : {best_epoch}")
print(f"Best Macro F1 : {best_f1:.4f}")

In [ ]:
# ============================================================
# Load Best ResNet50 Checkpoint
# ============================================================

best_model = resnet50(
    weights=None
)

best_model.fc = nn.Linear(
    best_model.fc.in_features,
    NUM_CLASSES
)

best_model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE
    )
)

best_model = best_model.to(DEVICE)
best_model.eval()

print("Best ResNet50 checkpoint loaded successfully.")
print("Checkpoint path:")
print(BEST_MODEL_PATH)

In [ ]:
# ============================================================
# Evaluate Best Model on Test Set
# ============================================================

best_model.eval()

test_running_loss = 0.0
test_total_samples = 0

all_test_labels = []
all_test_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = best_model(images)

        loss = criterion(
            outputs,
            labels
        )

        batch_size = images.size(0)

        test_running_loss += (
            loss.item() * batch_size
        )

        test_total_samples += batch_size

        predictions = outputs.argmax(
            dim=1
        )

        all_test_labels.extend(
            labels.cpu().numpy()
        )

        all_test_predictions.extend(
            predictions.cpu().numpy()
        )


test_loss = (
    test_running_loss /
    test_total_samples
)

test_accuracy = accuracy_score(
    all_test_labels,
    all_test_predictions
)

test_macro_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="weighted",
    zero_division=0
)

print("=" * 65)
print("FINAL TEST RESULTS")
print("=" * 65)

print(f"Test Loss        : {test_loss:.4f}")
print(f"Test Accuracy    : {test_accuracy * 100:.2f}%")
print(f"Test Macro F1    : {test_macro_f1:.4f}")
print(f"Test Weighted F1 : {test_weighted_f1:.4f}")

In [ ]:
# ============================================================
# Detailed Classification Report
# ============================================================

target_names = (
    class_info
    .sort_values("class_index")["class_name"]
    .tolist()
)

report = classification_report(
    all_test_labels,
    all_test_predictions,
    target_names=target_names,
    digits=4,
    zero_division=0
)

print("=" * 80)
print("RESNET50 FROM SCRATCH - CLASSIFICATION REPORT")
print("=" * 80)

print(report)

In [ ]:
# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(
    all_test_labels,
    all_test_predictions
)

plt.figure(figsize=(10, 8))

plt.imshow(cm, interpolation="nearest")

plt.title(
    "ResNet50 From Scratch - Confusion Matrix"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    ticks=np.arange(NUM_CLASSES),
    labels=target_names,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(NUM_CLASSES),
    labels=target_names
)

plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Normalized Confusion Matrix
# ============================================================

cm_normalized = (
    cm.astype("float")
    / cm.sum(axis=1)[:, np.newaxis]
)

plt.figure(figsize=(10, 8))

plt.imshow(
    cm_normalized,
    interpolation="nearest",
    vmin=0,
    vmax=1
)

plt.title(
    "ResNet50 From Scratch - Normalized Confusion Matrix"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    ticks=np.arange(NUM_CLASSES),
    labels=target_names,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(NUM_CLASSES),
    labels=target_names
)

plt.colorbar()

for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):

        plt.text(
            j,
            i,
            f"{cm_normalized[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Per-Class Performance Table
# ============================================================

report_dict = classification_report(
    all_test_labels,
    all_test_predictions,
    target_names=target_names,
    output_dict=True,
    zero_division=0
)

per_class_results = pd.DataFrame(
    report_dict
).transpose()

per_class_results = per_class_results.loc[
    target_names,
    ["precision", "recall", "f1-score", "support"]
]

per_class_results = per_class_results.reset_index()

per_class_results.columns = [
    "Class",
    "Precision",
    "Recall",
    "F1_Score",
    "Support"
]

per_class_results["Support"] = (
    per_class_results["Support"].astype(int)
)

per_class_results

In [ ]:
# ============================================================
# Save Final Test Results
# ============================================================

overall_results = {
    "experiment": "ResNet50_Scratch",
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "best_validation_macro_f1": float(best_val_f1),
    "best_epoch": int(best_epoch),
    "total_test_samples": int(len(all_test_labels))
}

OVERALL_RESULTS_PATH = (
    RESULTS_DIR /
    "resnet50_scratch_test_results.json"
)

PER_CLASS_RESULTS_PATH = (
    RESULTS_DIR /
    "resnet50_scratch_per_class_results.csv"
)

with open(OVERALL_RESULTS_PATH, "w") as file:
    json.dump(
        overall_results,
        file,
        indent=4
    )

per_class_results.to_csv(
    PER_CLASS_RESULTS_PATH,
    index=False
)

print("=" * 65)
print("FINAL RESULTS SAVED")
print("=" * 65)

print("\nOverall results:")
print(OVERALL_RESULTS_PATH)

print("\nPer-class results:")
print(PER_CLASS_RESULTS_PATH)

In [ ]:
# ============================================================
# Save Predictions and Confusion Matrices
# ============================================================

prediction_results = test_df[
    [
        "image",
        "class_index",
        "class_code",
        "class_name"
    ]
].copy()

prediction_results["predicted_index"] = all_test_predictions

prediction_results["predicted_class"] = [
    class_info.iloc[index]["class_name"]
    for index in all_test_predictions
]

prediction_results["correct_prediction"] = (
    prediction_results["class_index"]
    ==
    prediction_results["predicted_index"]
)

PREDICTIONS_PATH = (
    RESULTS_DIR /
    "resnet50_scratch_test_predictions.csv"
)

RAW_CM_PATH = (
    RESULTS_DIR /
    "resnet50_scratch_confusion_matrix.csv"
)

NORMALIZED_CM_PATH = (
    RESULTS_DIR /
    "resnet50_scratch_confusion_matrix_normalized.csv"
)


prediction_results.to_csv(
    PREDICTIONS_PATH,
    index=False
)

pd.DataFrame(
    cm,
    index=target_names,
    columns=target_names
).to_csv(
    RAW_CM_PATH
)

pd.DataFrame(
    cm_normalized,
    index=target_names,
    columns=target_names
).to_csv(
    NORMALIZED_CM_PATH
)

print("=" * 65)
print("ADDITIONAL RESULTS SAVED")
print("=" * 65)

print("\nTest predictions:")
print(PREDICTIONS_PATH)

print("\nRaw confusion matrix:")
print(RAW_CM_PATH)

print("\nNormalized confusion matrix:")
print(NORMALIZED_CM_PATH)

print(
    "\nCorrect test predictions:",
    prediction_results["correct_prediction"].sum()
)

print(
    "Incorrect test predictions:",
    (~prediction_results["correct_prediction"]).sum()
)

In [ ]:
# ============================================================
# Visualize Misclassified Test Images
# ============================================================

misclassified_df = prediction_results[
    prediction_results["correct_prediction"] == False
].copy()

print(
    "Total misclassified test images:",
    len(misclassified_df)
)

sample_errors = misclassified_df.sample(
    n=min(8, len(misclassified_df)),
    random_state=RANDOM_SEED
)

fig, axes = plt.subplots(
    2,
    4,
    figsize=(16, 9)
)

axes = axes.flatten()

for ax, (_, row) in zip(
    axes,
    sample_errors.iterrows()
):

    original_row = test_df[
        test_df["image"] == row["image"]
    ].iloc[0]

    image_path = original_row[
        "cached_image_path"
    ]

    with Image.open(image_path) as image:
        image = image.convert("RGB")

        ax.imshow(image)

    ax.set_title(
        f"True: {row['class_name']}\n"
        f"Pred: {row['predicted_class']}"
    )

    ax.axis("off")

plt.suptitle(
    "ResNet50 From Scratch - Misclassified Test Images",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Final ResNet50 Scratch Experiment Summary
# ============================================================

total_training_minutes = (
    history_df["epoch_time_seconds"].sum() / 60
)

final_summary = pd.DataFrame([
    {
        "Experiment": "ResNet50 Scratch",
        "Architecture": "ResNet50",
        "Transfer_Learning": False,
        "Data_Augmentation": False,
        "Image_Size": IMAGE_SIZE,
        "Batch_Size": BATCH_SIZE,
        "Epochs_Completed": len(history_df),
        "Best_Epoch": best_epoch,
        "Best_Validation_Macro_F1": best_val_f1,
        "Test_Loss": test_loss,
        "Test_Accuracy": test_accuracy,
        "Test_Macro_F1": test_macro_f1,
        "Test_Weighted_F1": test_weighted_f1,
        "Training_Time_Minutes": total_training_minutes,
        "Total_Parameters": total_params,
        "Trainable_Parameters": trainable_params
    }
])

SUMMARY_PATH = (
    RESULTS_DIR /
    "resnet50_scratch_summary.csv"
)

final_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print("=" * 70)
print("RESNET50 FROM SCRATCH - FINAL SUMMARY")
print("=" * 70)

print(f"Epochs completed          : {len(history_df)}")
print(f"Best epoch                : {best_epoch}")
print(f"Best Validation Macro F1 : {best_val_f1:.4f}")

print("\nTEST PERFORMANCE")
print(f"Test Loss                 : {test_loss:.4f}")
print(f"Test Accuracy             : {test_accuracy * 100:.2f}%")
print(f"Test Macro F1             : {test_macro_f1:.4f}")
print(f"Test Weighted F1          : {test_weighted_f1:.4f}")

print("\nMODEL / TRAINING")
print(f"Total parameters          : {total_params:,}")
print(f"Trainable parameters      : {trainable_params:,}")
print(f"Training time             : {total_training_minutes:.2f} min")

print("\nSummary saved:")
print(SUMMARY_PATH)